# Dataset + Augmentation

In [31]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader


class OralBinaryDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        for label_name in ["oral", "non_oral"]:
            label = 1 if label_name == "oral" else 0
            base_dir = os.path.join(root, label_name)

            for subdir, _, files in os.walk(base_dir):
                for f in files:
                    if f.lower().endswith((".jpg", ".png", ".jpeg")):
                        self.samples.append((os.path.join(subdir, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
    
        img = cv2.imread(path)
    
        # ❗ Fix ảnh lỗi
        if img is None:
            return self.__getitem__((idx + 1) % len(self))
    
        # ❗ Fix grayscale
        if len(img.shape) == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
        # ❗ Nếu có transform thì apply thêm
        if self.transform:
            img = self.transform(image=img)["image"]
    
        img = np.transpose(img, (2, 0, 1))
    
        return torch.tensor(img).float(), torch.tensor(label)

In [32]:
import albumentations as A

train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Blur(p=0.2),
    A.Normalize()
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize()
])

# Balance dataset (QUAN TRỌNG)

In [6]:
# from torch.utils.data import random_split

# dataset = OralBinaryDataset("/kaggle/input/datasets/nguynhongphmkhi/oral-classification")

# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size

# train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [33]:
dataset_path="/kaggle/input/datasets/nguynhongphmkhi/oral-classification"

In [34]:
from sklearn.model_selection import train_test_split

full_dataset = OralBinaryDataset(dataset_path, transform=None)

labels = [label for _, label in full_dataset.samples]

train_idx, val_idx = train_test_split(
    range(len(labels)),
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [35]:
train_dataset = OralBinaryDataset(dataset_path, transform=train_transform)
val_dataset   = OralBinaryDataset(dataset_path, transform=val_transform)

train_dataset.samples = [full_dataset.samples[i] for i in train_idx]
val_dataset.samples   = [full_dataset.samples[i] for i in val_idx]

In [36]:
from torch.utils.data import WeightedRandomSampler

train_labels = [label for _, label in train_dataset.samples]

class_counts = np.bincount(train_labels)
class_weights = 1. / class_counts

sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    sampler=sampler,
    num_workers=0
)

In [38]:
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [39]:
def check_balance(loader):
    oral = 0
    non_oral = 0

    for imgs, labels in loader:
        oral += (labels == 1).sum().item()
        non_oral += (labels == 0).sum().item()

    print("Oral:", oral, "Non-oral:", non_oral)

In [41]:
imgs, labels = next(iter(train_loader))
print(imgs.shape)

torch.Size([32, 3, 224, 224])


In [40]:
imgs, labels = next(iter(val_loader))
print(imgs.shape)

torch.Size([32, 3, 224, 224])


In [42]:
check_balance(train_loader)

libpng warning: iCCP: extra compressed data
libpng warning: iCCP: extra compressed data
libpng warning: iCCP: extra compressed data


Oral: 2993 Non-oral: 3077


In [43]:
check_balance(val_loader)

Oral: 293 Non-oral: 1225


In [ ]:
# from torch.utils.data import DataLoader, random_split

# dataset = OralDataset(oral_dir, non_oral_dir)

# train_size = int(0.8 * len(dataset))
# val_size = len(dataset) - train_size

# train_ds, val_ds = random_split(dataset, [train_size, val_size])

# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=32)

In [ ]:
max_non_oral = len([l for l in labels if l == 1]) * 2

filtered_samples = []
count_non = 0

for path, label in dataset.samples:
    if label == 0:
        if count_non < max_non_oral:
            filtered_samples.append((path, label))
            count_non += 1
    else:
        filtered_samples.append((path, label))

dataset.samples = filtered_samples

# Model (FAST + đủ mạnh)

In [21]:
import timm
import torch.nn as nn

model = timm.create_model('mobilenetv3_small_100', pretrained=True)
model.classifier = nn.Linear(model.classifier.in_features, 2)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

model.safetensors:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

MobileNetV3(
  (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): Hardswish()
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=16, bias=False)
        (bn1): BatchNormAct2d(
          16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): ReLU(inplace=True)
          (conv_expand): Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
          (gate): Hardsigmoid()
        )
        (conv_pw): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2): B

In [44]:
weights = torch.tensor([1.0, 2.5]).to(device)  # non_oral, oral
criterion = nn.CrossEntropyLoss(weight=weights)

In [45]:
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [46]:
def evaluate_metrics(probs, labels, threshold=0.7):
    preds = (probs > threshold).astype(int)

    TP = ((preds == 1) & (labels == 1)).sum()
    FP = ((preds == 1) & (labels == 0)).sum()
    FN = ((preds == 0) & (labels == 1)).sum()

    precision = TP / (TP + FP + 1e-6)
    recall = TP / (TP + FN + 1e-6)

    return precision, recall

In [47]:
import matplotlib.pyplot as plt

def plot_training(history):
    epochs = range(len(history["loss"]))

    plt.figure()
    plt.plot(epochs, history["loss"], label="Loss")
    plt.plot(epochs, history["precision"], label="Precision")
    plt.plot(epochs, history["recall"], label="Recall")

    plt.legend()
    plt.title("Training Metrics")
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.show()

In [48]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import torch # Ajout de l'import torch
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def plot_confusion(probs, labels, threshold=0.7):
    # Conversion PyTorch vers NumPy sécurisée
    if isinstance(probs, torch.Tensor):
        probs = probs.detach().cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.detach().cpu().numpy()
    
    preds = (probs > threshold).astype(int)

    cm = confusion_matrix(labels, preds)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d")

    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

In [49]:
def validate_with_paths(model, dataset):
    model.eval()

    probs_all = []
    labels_all = []
    paths_all = []

    with torch.no_grad():
        for i in range(len(dataset)):
            img, label = dataset[i]
            path, _ = dataset.samples[i]

            img = img.unsqueeze(0).to(device)

            out = model(img)
            prob = torch.softmax(out, dim=1)[0][1].item()

            probs_all.append(prob)
            labels_all.append(label.item())
            paths_all.append(path)

    return probs_all, labels_all, paths_all

In [50]:
import cv2

def show_false_positive(probs, labels, paths, threshold=0.7, max_show=5):
    count = 0

    for prob, label, path in zip(probs, labels, paths):
        if prob > threshold and label == 0:  # FP
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            plt.figure()
            plt.imshow(img)
            plt.title(f"FP | prob={prob:.2f}")
            plt.axis("off")
            plt.show()

            count += 1
            if count >= max_show:
                break

In [51]:
def show_false_negative(probs, labels, paths, threshold=0.7, max_show=5):
    count = 0

    for prob, label, path in zip(probs, labels, paths):
        if prob <= threshold and label == 1:  # FN
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            plt.figure()
            plt.imshow(img)
            plt.title(f"FN | prob={prob:.2f}")
            plt.axis("off")
            plt.show()

            count += 1
            if count >= max_show:
                break

# Training

In [52]:
import torch.nn.functional as F

def validate(model, loader):
    model.eval()

    probs_all = []
    labels_all = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)

            outputs = model(imgs)
            probs = F.softmax(outputs, dim=1)[:,1]

            probs_all.extend(probs.cpu().numpy())
            labels_all.extend(labels.numpy())

    return np.array(probs_all), np.array(labels_all)

In [53]:
from tqdm import tqdm
import torch.nn.functional as F

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0

    for imgs, labels in tqdm(loader):
        imgs = imgs.to(device)
        labels = labels.to(device)

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
best_precision = 0
patience = 3
counter = 0

history = {
    "loss": [],
    "precision": [],
    "recall": []
}
for epoch in range(20):

    train_loss = train_one_epoch(model, train_loader)

    probs, labels = validate(model, val_loader)

    precision, recall = evaluate_metrics(probs, labels, threshold=0.7)
    history["loss"].append(train_loss)
    history["precision"].append(precision)
    history["recall"].append(recall)

    print(f"Epoch {epoch}")
    print(f"Loss: {train_loss:.4f}")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}")

    #  Early stopping theo precision
    if precision > best_precision:
        best_precision = precision
        counter = 0

        torch.save(model.state_dict(), "best_model.pth")
        print("Done: Save best model")

    else:
        counter += 1

    # if counter >= patience:
    #     print("Done: Early stopping")
    #     break

100%|██████████| 190/190 [01:46<00:00,  1.78it/s]


Epoch 0
Loss: 0.0302
Precision: 1.0000, Recall: 0.9556
Done: Save best model


100%|██████████| 190/190 [01:48<00:00,  1.75it/s]


Epoch 1
Loss: 0.0095
Precision: 0.9899, Recall: 1.0000


100%|██████████| 190/190 [01:49<00:00,  1.74it/s]


...

100%|██████████| 190/190 [01:48<00:00,  1.75it/s]


Epoch 4
Loss: 0.0025
Precision: 0.9966, Recall: 1.0000


100%|██████████| 190/190 [01:50<00:00,  1.72it/s]


Epoch 5
Loss: 0.0043
Precision: 0.9865, Recall: 1.0000


 16%|█▌        | 30/190 [00:17<01:35,  1.68it/s]libpng warning: iCCP: extra compressed data


In [ ]:
probs, labels, paths = validate_with_paths(model, val_dataset)

plot_training(history)
plot_confusion(probs, labels)

show_false_positive(probs, labels, paths)
show_false_negative(probs, labels, paths)

In [ ]:
probs, labels, paths = validate_with_paths(model, train_dataset)

plot_training(history)
plot_confusion(probs, labels)

show_false_positive(probs, labels, paths)
show_false_negative(probs, labels, paths)

# Tune threshold (rất quan trọng cho UNet)

# Save model

In [ ]:
torch.save(model.state_dict(), "oral_classifier.pth")

In [ ]:
model.load_state_dict(torch.load("oral_classifier.pth"))
model.eval()

8.2 Gate function

In [ ]:
def is_oral(img, threshold=0.7):
    model.eval()

    img = cv2.resize(img, (224,224))
    img = transform(image=img)["image"]
    img = np.transpose(img, (2,0,1))

    img = torch.tensor(img).float().unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(img)
        prob = torch.softmax(out, dim=1)[0][1].item()

    return prob > threshold, prob

# FULL PIPELINE

In [ ]:
def pipeline(image):
    ok, prob = is_oral(image)

    if ok:
        mask = unet_model(image)   # bạn đã có
        return mask, prob
    else:
        return None, prob